# Datathon CDIA 2026 — Pipeline de Dados
## Raio-X do Mercado de Trabalho e Vagas Industriais em Santa Catarina

**Pergunta central de negócio**

> *"Quais regiões e setores industriais de SC enfrentam apagão de mão de obra ou
> expansão acelerada, e como renda e escolaridade explicam esses gargalos?"*

Este notebook implementa o **pipeline completo** (Entregável 1):

1. Ingestão e validação das bases
2. Limpeza e filtragem (exclusões MOV × EX)
3. Cruzamento (join) CAGED × municípios
4. Métricas de negócio (saldo, saldo por 10 mil hab., turnover)
5. Agregações territoriais, setoriais e socioeconômicas
6. Enriquecimento com IA — agrupamento de CBOs em macrofamílias
7. Exportação das tabelas-base para o dashboard

Todas as tabelas finais são gravadas em `outputs/` e consumidas pelo dashboard (Entregável 2).


## 0. Configuração e dependências

```bash
pip install pandas pyarrow openpyxl plotly
```
Para o passo de IA (opcional), defina a variável de ambiente `OPENAI_API_KEY`.


In [1]:
import os
import unicodedata
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# Caminhos (ajuste se executar de outro diretório)
BASE = ".." if os.path.basename(os.getcwd()) == "notebooks" else "."
DATASET = os.path.join(BASE, "data")  # data/raw + data/reference
OUTPUTS = os.path.join(BASE, "outputs")
os.makedirs(OUTPUTS, exist_ok=True)

CAGED_PARQUET = os.path.join(DATASET, "raw", "Novo_Caged_MOV_Limpo.parquet")
MUNIC_CSV     = os.path.join(DATASET, "reference", "Municipios_SC_limpo.csv")
DICIONARIO    = os.path.join(DATASET, "reference", "Dicionario_de_dados.xlsx")

print("data:", os.path.abspath(DATASET))
print("outputs:", os.path.abspath(OUTPUTS))

data: /projects/sandbox/datathon/data
outputs: /projects/sandbox/datathon/outputs


## 1. Ingestão e validação

O CAGED está em **Parquet (zstd)** — ~42 MB, 3.397.416 movimentações, 30 colunas,
somente Santa Catarina (uf=42). Lemos os dicionários para traduzir os códigos.


In [2]:
# --- CAGED (movimentações) ---
caged = pd.read_parquet(CAGED_PARQUET)
print("CAGED:", caged.shape)

# --- Municípios (demografia / socioeconômico) ---
munic = pd.read_csv(MUNIC_CSV)
print("Municípios:", munic.shape)

# --- Dicionário de dados: cada aba mapeia Código -> Descrição ---
xls = pd.ExcelFile(DICIONARIO)
def tabela_dic(aba):
    d = pd.read_excel(DICIONARIO, sheet_name=aba, dtype=str)
    d.columns = [c.strip().lower() for c in d.columns]
    return dict(zip(d["código"].astype(str).str.strip(),
                    d["descrição"].astype(str).str.strip()))

MAP_SECAO   = tabela_dic("seção")
MAP_INSTR   = tabela_dic("graudeinstrução")
MAP_TIPOMOV = tabela_dic("tipomovimentação")
MAP_MUNIC   = tabela_dic("município")
print("Abas do dicionário:", xls.sheet_names)

CAGED: (3397416, 30)
Municípios: (296, 15)


Abas do dicionário: ['Layout', 'região', 'uf', 'município', 'seção', 'subclasse', 'categoria', 'cbo2002ocupação', 'graudeinstrução', 'raçacor', 'sexo', 'tipoempregador', 'tipoestabelecimento', 'tipomovimentação', 'tipodedeficiência', 'indtrabintermitente', 'indtrabparcial', 'tamestabjan', 'indicadoraprendiz', 'origemdainformação', 'indicadordeexclusão', 'indicadordeforadoprazo', 'unidadesaláriocódigo']


In [3]:
# Validação de schema mínima
esperadas = {"competênciamov","uf","município","seção","saldomovimentação",
             "cbo2002ocupação","graudeinstrução","salário","tipomovimentação"}
faltando = esperadas - set(caged.columns)
assert not faltando, f"Colunas ausentes no CAGED: {faltando}"
print("Schema OK. UFs presentes:", caged["uf"].unique())
print("Competências (meses):", caged["competênciamov"].nunique(),
      "de", caged["competênciamov"].min(), "a", caged["competênciamov"].max())

Schema OK. UFs presentes: <ArrowStringArray>
['42']
Length: 1, dtype: str
Competências (meses): 24 de 202409 a 202608


## 2. Limpeza, tipagem e filtragem (exclusões MOV × EX)

No Parquet todas as colunas vêm como **texto** (preserva fidelidade). Aqui convertemos
os tipos e aplicamos o filtro de exclusões exigido pelo guia.

**Sobre o filtro MOV × EX:** a base de movimentações já é a "MOV_Limpo" (as exclusões
da base EX já foram removidas na origem). Mantemos a verificação defensiva: se existir a
coluna `indicadordeexclusão`, descartamos registros marcados como excluídos (`=1`).


In [4]:
df = caged.copy()

# Colunas numéricas (salário usa vírgula decimal e ponto de milhar)
def to_float_br(s):
    return pd.to_numeric(
        s.astype(str).str.replace(".", "", regex=False).str.replace(",", ".", regex=False),
        errors="coerce")

df["saldo"]          = pd.to_numeric(df["saldomovimentação"], errors="coerce")
df["salario"]        = to_float_br(df["salário"])
df["salario_fixo"]   = to_float_br(df.get("valorsaláriofixo", pd.Series(index=df.index)))
df["idade"]          = pd.to_numeric(df["idade"], errors="coerce")
df["grau_cod"]       = pd.to_numeric(df["graudeinstrução"], errors="coerce")
# Escolaridade: os códigos NÃO são uma escala numérica (80 = pós-graduação fica entre
# 9 = superior e 10 = mestrado; 99 = não identificado). Convertemos para a posição
# ordinal 1..12 e deixamos 99 de fora, em vez de tirar a média dos códigos.
ORDEM_INSTR = ["1", "2", "3", "4", "5", "6", "7", "8", "9", "80", "10", "11"]
_g = df["graudeinstrução"].astype(str).str.strip().str.replace(r"\.0$", "", regex=True)
df["grau_ord"]       = _g.map({c: i + 1 for i, c in enumerate(ORDEM_INSTR)})   # 99 -> NaN
df["superior"]       = _g.isin(["9", "80", "10", "11"]).astype(float).where(df["grau_ord"].notna())
df["mun_cod"]        = df["município"].astype(str).str.strip()

# Filtro de exclusões (defensivo)
if "indicadordeexclusão" in df.columns:
    antes = len(df)
    df = df[df["indicadordeexclusão"].astype(str).str.strip() != "1"].copy()
    print(f"Exclusões removidas: {antes - len(df)}")
else:
    print("Sem coluna de exclusão (base já limpa na origem).")

# Descartar artefatos de índice
df = df.drop(columns=[c for c in ["Unnamed: 0.1", "Unnamed: 0"] if c in df.columns])

# Traduções legíveis
df["setor"]      = df["seção"].map(MAP_SECAO).fillna("Não identificado")
df["grau_instr"] = df["graudeinstrução"].astype(str).map(MAP_INSTR).fillna("Não Identificado")
df["tipo_mov"]   = df["tipomovimentação"].astype(str).map(MAP_TIPOMOV).fillna("Não Identificado")

df["admissao"]     = (df["saldo"] > 0).astype(int)
df["desligamento"] = (df["saldo"] < 0).astype(int)

print("Linhas após limpeza:", len(df))
df[["competênciamov","mun_cod","setor","saldo","salario","grau_instr"]].head()

Sem coluna de exclusão (base já limpa na origem).


Linhas após limpeza: 3397416


,competênciamov,mun_cod,setor,saldo,salario,grau_instr
0,202509,420890,Construção,1,2018.00,Médio Completo
1,202509,421605,"Comércio, Reparação de Veículos Automotores e ...",-1,1693.35,Médio Completo
2,202509,420820,Construção,1,2556.00,Médio Incompleto
3,202509,420820,"Comércio, Reparação de Veículos Automotores e ...",-1,3181.34,Superior Completo
4,202509,421555,Indústrias de Transformação,-1,1650.00,Superior Incompleto


## 3. Cruzamento CAGED × municípios

Chave: código do município. No `Municipios_SC_limpo.csv` o código IBGE tem 7 dígitos;
no CAGED, 6 (sem o dígito verificador). Alinhamos pelos **6 primeiros dígitos**.


In [5]:
# Normaliza nomes de colunas do arquivo de municípios
munic.columns = [c.strip() for c in munic.columns]

def achar_col(cols, *chaves):
    for c in cols:
        low = c.lower()
        if all(k in low for k in chaves):
            return c
    return None

col_codigo = achar_col(munic.columns, "código")
col_nome   = achar_col(munic.columns, "município")
col_pop    = achar_col(munic.columns, "população", "último censo") or achar_col(munic.columns, "população")
col_idhm   = achar_col(munic.columns, "idhm")
col_pib    = achar_col(munic.columns, "pib per capita") or achar_col(munic.columns, "pib")
col_escol  = achar_col(munic.columns, "escolariza")

print("Mapeadas:", dict(codigo=col_codigo, nome=col_nome, pop=col_pop,
                         idhm=col_idhm, pib=col_pib, escol=col_escol))

mun = pd.DataFrame({
    "mun_cod":   munic[col_codigo].astype(str).str.replace(r"\D", "", regex=True).str[:6],
    "municipio": munic[col_nome].astype(str).str.strip(),
})
def num(col):
    if col is None: return np.nan
    return to_float_br(munic[col]) if munic[col].dtype == object else pd.to_numeric(munic[col], errors="coerce")

mun["populacao"]     = num(col_pop)
mun["idhm"]          = num(col_idhm)
mun["pib_per_capita"]= num(col_pib)
mun["escolarizacao"] = num(col_escol)
mun = mun.drop_duplicates("mun_cod")
print("Municípios normalizados:", mun.shape)
mun.head()

Mapeadas: {'codigo': 'Código [-]', 'nome': 'Município [-]', 'pop': 'População no último censo - pessoas [2022]', 'idhm': 'IDHM <span>Índice de desenvolvimento humano municipal</span> [2010]', 'pib': 'PIB per capita - R$ [2023]', 'escol': 'Escolarização <span>6 a 14 anos</span> - % [2022]'}
Municípios normalizados: (296, 6)


,mun_cod,municipio,populacao,idhm,pib_per_capita,escolarizacao
0,420005,Abdon Batista,2598.0,0.694,44644.04,100.00
1,420010,Abelardo Luz,17392.0,0.696,62434.34,99.69
2,420020,Agrolândia,10990.0,0.725,45141.98,99.47
3,420030,Agronômica,6055.0,0.741,46144.62,98.98
4,420040,Água Doce,6508.0,0.698,114906.54,99.03


In [6]:
dfm = df.merge(mun, on="mun_cod", how="left")
sem_match = dfm["municipio"].isna().mean()
print(f"% movimentações sem município correspondente: {sem_match:.2%}")
dfm[["mun_cod","municipio","populacao","setor","saldo"]].head()

% movimentações sem município correspondente: 0.00%


,mun_cod,municipio,populacao,setor,saldo
0,420890,Jaraguá do Sul,182660.0,Construção,1
1,421605,São Cristóvão do Sul,6084.0,"Comércio, Reparação de Veículos Automotores e ...",-1
2,420820,Itajaí,264054.0,Construção,1
3,420820,Itajaí,264054.0,"Comércio, Reparação de Veículos Automotores e ...",-1
4,421555,Santa Helena,2425.0,Indústrias de Transformação,-1


## 4. Métricas de negócio

- **Saldo** = Σ(saldomovimentação) = admissões − desligamentos.
- **Saldo por 10.000 habitantes** = saldo / população × 10.000 (normalização exigida pelo guia).
- **Turnover (rotatividade)** = (admissões + desligamentos) / 2 ÷ estoque aproximado.
  Como não temos o estoque de vínculos, usamos a **taxa de giro** = (adm+desl) por 10 mil hab,
  que diferencia "apagão" (giro alto, saldo baixo) de "expansão" (saldo alto).


### 4.1 Medidas salariais (metodologia — etapa 3.2)

Ver `docs/metodologia_salarial.md`. Decisões aplicadas aqui, **sem sobrescrever** a
coluna de origem `salario` nem a `salario_medio` histórica:

- `salario` (origem) **já é mensalizado pelo MTE** — não convertemos periodicidades.
- Medida **principal**: **mediana excluindo zeros** (`salario_mediano_s0`), estável a
  zeros (33.912 registros) e extremos (535 acima de R$ 50 mil).
- Medida **complementar**, sempre rotulada: a média. Mantemos `salario_medio`
  (média de todos os registros, inclui zeros) para compatibilidade e rastreabilidade,
  e acrescentamos `salario_medio_s0` (média excluindo zeros).
- **Rastreabilidade**: `n_salario` (registros com salário não nulo) e `n_salario_s0`
  (registros com salário > 0, a base da medida principal).

`salario > 0` isola os registros com remuneração declarada; os zeros ficam contados
em `n_salario - n_salario_s0`, nunca descartados silenciosamente.


In [7]:
# Agregações salariais da metodologia 3.2 (ver docs/metodologia_salarial.md).
# A coluna de origem `salario` é preservada; nada é sobrescrito. s0 = "sem zeros" (salario > 0).
def _median_pos(s):
    v = s[s > 0]
    return v.median() if len(v) else np.nan

def _mean_pos(s):
    v = s[s > 0]
    return v.mean() if len(v) else np.nan

SAL_AGG = dict(
    salario_medio=("salario", "mean"),             # media, inclui zeros (historico; complemento)
    salario_mediano=("salario", "median"),         # mediana, inclui zeros (historico)
    salario_medio_s0=("salario", _mean_pos),       # media sem zeros (complemento rotulado)
    salario_mediano_s0=("salario", _median_pos),   # MEDIDA PRINCIPAL: mediana sem zeros
    n_salario=("salario", "count"),                # registros com salario nao nulo
    n_salario_s0=("salario", lambda s: int((s > 0).sum())),  # registros com salario > 0 (base da principal)
)

In [8]:
def agrega(grupo_cols, nome_pop="populacao"):
    g = dfm.groupby(grupo_cols, dropna=False).agg(
        admissoes=("admissao", "sum"),
        desligamentos=("desligamento", "sum"),
        saldo=("saldo", "sum"),
        movimentacoes=("saldo", "size"),
        grau_medio=("grau_ord", "mean"),       # escala ordinal 1..12 (ver seção 2)
        pct_superior=("superior", "mean"),    # fração com superior completo ou mais
        idade_media=("idade", "mean"),
        **SAL_AGG,                             # medidas salariais da metodologia 3.2 (seção 4.1)
    ).reset_index()
    g["giro"] = g["admissoes"] + g["desligamentos"]
    return g

# Base populacional por município (para normalizar)
pop_mun = mun.set_index("mun_cod")["populacao"]

## 5. Eixo territorial — municípios em expansão vs. apagão

In [9]:
terr = agrega(["mun_cod", "municipio"])
terr = terr.merge(mun[["mun_cod","populacao","idhm","pib_per_capita","escolarizacao"]],
                  on="mun_cod", how="left")
terr["saldo_por_10k"] = terr["saldo"] / terr["populacao"] * 10_000
terr["giro_por_10k"]  = terr["giro"]  / terr["populacao"] * 10_000

print("TOP 10 EXPANSÃO (saldo por 10 mil hab.):")
display_cols = ["municipio","saldo","saldo_por_10k","giro_por_10k","salario_medio","grau_medio"]
print(terr.sort_values("saldo_por_10k", ascending=False)[display_cols].head(10).to_string(index=False))

print("\nTOP 10 RETRAÇÃO / APAGÃO (menor saldo por 10 mil hab.):")
print(terr.sort_values("saldo_por_10k")[display_cols].head(10).to_string(index=False))

TOP 10 EXPANSÃO (saldo por 10 mil hab.):
                municipio  saldo  saldo_por_10k  giro_por_10k  salario_medio  grau_medio
   Governador Celso Ramos   1571     928.761454   8728.938812    2156.807258    6.550830
                 Guatambú    581     689.614243   8195.845697    2262.959445    4.503983
Santo Amaro da Imperatriz   1282     470.079202   5021.267234    2390.745237    6.954287
               Navegantes   3514     406.708256   5427.946436    2499.209799    6.503220
             Treze Tílias    344     391.487425   6680.323205    2441.378600    6.350596
             Nova Erechim    190     368.574200   2750.727449    2386.794238    6.157969
      Bom Jardim da Serra    141     350.223547   5233.482365    1924.389929    6.465591
                  Vargeão    123     338.470006   5335.718217    2437.136668    6.204745
           Pedras Grandes    129     303.886926   4487.632509    2395.816157    6.655118
           Otacílio Costa    481     277.841959   4444.893715    2825

### 5.1 Enriquecimento com bases públicas do IBGE

O CAGED mede **fluxo** (admissões e desligamentos), não **estoque** de empregos. Para
normalizar melhor do que só pela população, cruzamos com bases oficiais do IBGE baixadas por
`src/enrichment/ibge.py` (APIs públicas, versionadas em `data/external/`):

| Base | Uso |
|---|---|
| **CEMPRE 2024** (SIDRA 9509) — pessoal ocupado assalariado | denominador do emprego formal: **saldo / estoque** e **taxa de rotatividade** |
| **PIB dos Municípios** (SIDRA 5938) — PIB 2023 e VAB por atividade 2021 | peso da **indústria** no valor adicionado municipal |
| **Divisão regional** (API de Localidades) | agregação por **região geográfica intermediária e imediata** |
| **Malha municipal** (API de Malhas) | mapa coroplético no dashboard |

Métricas novas:
- `cresc_emprego_pct` = saldo do CAGED / assalariados CEMPRE 2024 × 100 — quanto o emprego formal
  cresceu em 12 meses em relação ao estoque de partida;
- `rotatividade_pct` = mín(admissões, desligamentos) / assalariados × 100 — conceito do DIEESE:
  a parte do fluxo que é **reposição** (troca de trabalhadores) e não criação líquida de vagas.

> Premissa: o CEMPRE de 2024 é o estoque mais recente publicado e antecede o período do CAGED
> (set/2025–ago/2026). As taxas são aproximações e não substituem a RAIS.


In [10]:
EXTERNAL = os.path.join(BASE, "data", "external")
ibge = (pd.read_parquet(os.path.join(EXTERNAL, "ibge_regioes_sc.parquet"))
          .merge(pd.read_parquet(os.path.join(EXTERNAL, "ibge_cempre_2024.parquet")), on="codigo_ibge")
          .merge(pd.read_parquet(os.path.join(EXTERNAL, "ibge_pib_municipal.parquet")), on="codigo_ibge"))
ibge["mun_cod"] = ibge["codigo_ibge"].str[:6]          # mesma chave de 6 dígitos do CAGED
print("Municípios IBGE:", len(ibge), "| sem par no CAGED:",
      len(set(ibge["mun_cod"]) - set(terr["mun_cod"])))

terr = terr.merge(ibge.drop(columns=["nome_ibge"]), on="mun_cod", how="left")
terr["cresc_emprego_pct"] = terr["saldo"] / terr["assalariados"] * 100
terr["rotatividade_pct"]  = terr[["admissoes", "desligamentos"]].min(axis=1) / terr["assalariados"] * 100

# Mapa município -> região (os dois níveis) para calcular a mediana salarial por região
# diretamente dos microdados (a mediana não pode ser obtida ponderando medianas municipais).
_mun2reg = terr.set_index("mun_cod")[["regiao_intermediaria", "regiao_imediata"]]
dfm_reg = dfm.join(_mun2reg, on="mun_cod")

def agrega_regiao(nivel):
    soma = ["admissoes", "desligamentos", "saldo", "movimentacoes", "populacao", "assalariados",
            "pib_2023", "vab_total_2021", "vab_industria_2021", "n_salario", "n_salario_s0"]
    g = terr.groupby(nivel, as_index=False)[soma].sum()
    g = g.rename(columns={nivel: "regiao"}).assign(nivel=nivel)
    g["saldo_por_10k"]      = g["saldo"] / g["populacao"] * 10_000
    g["cresc_emprego_pct"]  = g["saldo"] / g["assalariados"] * 100
    g["rotatividade_pct"]   = g[["admissoes", "desligamentos"]].min(axis=1) / g["assalariados"] * 100
    g["part_industria_vab_2021"] = g["vab_industria_2021"] / g["vab_total_2021"]
    # salário médio e escolaridade ponderados pelo nº de movimentações de cada município
    for col in ["salario_medio", "salario_medio_s0", "grau_medio", "pct_superior"]:
        w = terr.dropna(subset=[col])
        media = w.groupby(nivel).apply(lambda d: np.average(d[col], weights=d["movimentacoes"]),
                                       include_groups=False).rename(col)
        g = g.merge(media, left_on="regiao", right_index=True, how="left")
    # medida principal: mediana sem zeros, calculada direto dos microdados por região
    med = (dfm_reg.groupby(nivel)["salario"].apply(_median_pos).rename("salario_mediano_s0"))
    g = g.merge(med, left_on="regiao", right_index=True, how="left")
    return g

regioes = pd.concat([agrega_regiao("regiao_intermediaria"), agrega_regiao("regiao_imediata")],
                    ignore_index=True)
print(regioes[regioes["nivel"] == "regiao_intermediaria"]
      [["regiao", "saldo", "cresc_emprego_pct", "rotatividade_pct", "part_industria_vab_2021"]]
      .sort_values("cresc_emprego_pct", ascending=False).to_string(index=False))

Municípios IBGE: 295 | sem par no CAGED: 0


       regiao  saldo  cresc_emprego_pct  rotatividade_pct  part_industria_vab_2021
      Caçador   1821           2.441248         70.131246                 0.355182
Florianópolis  12359           2.123102         54.111695                 0.134228
      Chapecó   7488           1.867523         55.008367                 0.278204
        Lages   1702           1.830797         62.035175                 0.230672
     Blumenau  10546           1.369609         68.929001                 0.247635
    Joinville   6244           1.217650         61.109107                 0.381912
     Criciúma   1322           0.427320         60.869186                 0.283887


## 6. Eixo setorial — setores industriais que lideram

Foco industrial = seções **B** (Extrativas), **C** (Transformação), **D** (Eletricidade/Gás),
**F** (Construção). Analisamos todos e sinalizamos os industriais.


In [11]:
SETORES_INDUSTRIAIS = {"B", "C", "D", "F"}
setor = agrega(["seção", "setor"])
setor["industrial"] = setor["seção"].isin(SETORES_INDUSTRIAIS)
setor = setor.sort_values("saldo", ascending=False)
print(setor[["setor","saldo","admissoes","desligamentos","giro",
             "salario_medio","grau_medio","industrial"]].to_string(index=False))

                                                            setor  saldo  admissoes  desligamentos   giro  salario_medio  grau_medio  industrial
             Atividades Administrativas e Serviços Complementares  11416     233612         222196 455808    2159.268844    6.608693       False
                 Atividades Profissionais, Científicas e Técnicas   6877      56443          49566 106009    2758.557769    7.251469       False
       Comércio, Reparação de Veículos Automotores e Motocicletas   5655     398899         393244 792143    2261.143947    6.735654       False
                                Transporte, Armazenagem e Correio   5290     111905         106615 218520    2454.730619    6.594618       False
                                  Saúde Humana e Serviços Sociais   4603      63262          58659 121921    2620.809884    7.321635       False
                                                       Construção   1737     137306         135569 272875    2820.729942    6.2515

In [12]:
# Cruzamento território × setor (onde cada setor cresce/encolhe)
terr_setor = agrega(["mun_cod","municipio","seção","setor"])
terr_setor = terr_setor.merge(pop_mun.rename("populacao"), on="mun_cod", how="left")
terr_setor["saldo_por_10k"] = terr_setor["saldo"] / terr_setor["populacao"] * 10_000
# Exemplo: maiores polos industriais por saldo
ind = terr_setor[terr_setor["seção"].isin(SETORES_INDUSTRIAIS)]
print("Maiores saldos industriais (município × setor):")
print(ind.sort_values("saldo", ascending=False)
        [["municipio","setor","saldo","salario_medio","grau_medio"]].head(15).to_string(index=False))

Maiores saldos industriais (município × setor):
         municipio                       setor  saldo  salario_medio  grau_medio
        Navegantes Indústrias de Transformação   2377    2389.959410    6.276112
         Joinville Indústrias de Transformação   1479    2897.099323    6.712828
           Chapecó Indústrias de Transformação   1296    2501.271174    5.691851
           Caçador                  Construção    736    3179.164068    6.286062
          Guatambú Indústrias de Transformação    561    2296.905658    4.197802
           Brusque                  Construção    531    2799.363855    6.198564
            Itajaí Indústrias de Transformação    508    2702.942842    6.581438
         Canoinhas                  Construção    441    1891.010391    6.240229
Balneário Camboriú                  Construção    341    3405.123510    6.441584
        Navegantes                  Construção    323    3128.288167    6.297866
          Camboriú                  Construção    319    2827

## 7. Eixo socioeconômico — renda e escolaridade explicam os gargalos?

Comparamos o perfil de **admitidos vs. desligados** por escolaridade e salário, e cruzamos
o saldo setorial/territorial com renda e escolaridade para apontar a causa-raiz:
desatratividade salarial, escolaridade não atendida ou compressão salarial.


In [13]:
# Perfil admitido vs desligado por escolaridade
perfil = (dfm.assign(mov=np.where(dfm["saldo"] > 0, "Admitido", "Desligado"))
            .groupby(["mov","grau_instr"], dropna=False)
            .agg(n=("saldo","size"), **SAL_AGG)
            .reset_index())
print(perfil.to_string(index=False))

# Salário por grau de instrução (há compressão salarial?). Medida principal: mediana sem
# zeros (salario_mediano_s0); média mantida como complemento. Ver docs/metodologia_salarial.md.
sal_grau = (dfm.groupby("grau_cod")
              .agg(grau=("grau_instr","first"),
                   grau_ord=("grau_ord","first"),
                   n=("salario","size"),
                   **SAL_AGG)
              .reset_index().sort_values("grau_ord", na_position="last"))
print("\nSalário por grau de instrução (mediana sem zeros × média):")
print(sal_grau[["grau","salario_mediano_s0","salario_medio","n","n_salario_s0"]].to_string(index=False))

      mov              grau_instr       n  salario_medio  salario_mediano  salario_medio_s0  salario_mediano_s0  n_salario  n_salario_s0
 Admitido 5ª Completo Fundamental   23242    2100.136684         1924.850       2107.936466            1926.150      23242         23156
 Admitido     6ª a 9ª Fundamental   76510    2171.500262         2043.590       2182.396557            2045.000      76510         76128
 Admitido              Analfabeto   15539    2227.480302         2156.230       2232.364813            2156.230      15539         15505
 Admitido       Até 5ª Incompleto   27075    2205.073698         2098.000       2215.959111            2100.000      27075         26942
 Admitido               Doutorado     721    6805.734619         4500.000       6950.332380            4600.000        721           706
 Admitido    Fundamental Completo  143155    2206.528698         2100.000       2220.176530            2100.000     143155        142275
 Admitido                Mestrado    1912


Salário por grau de instrução (mediana sem zeros × média):
                   grau  salario_mediano_s0  salario_medio       n  n_salario_s0
             Analfabeto            2156.230    2309.276050   30895         30770
      Até 5ª Incompleto            2105.600    2230.024712   52493         52205
5ª Completo Fundamental            1971.200    2145.044530   46664         46438
    6ª a 9ª Fundamental            2070.000    2204.825633  152357        151454
   Fundamental Completo            2112.000    2247.984994  286789        284747
       Médio Incompleto            2050.000    2075.640959  343854        341559
         Médio Completo            2155.000    2353.890719 2032882       2011335
    Superior Incompleto            2271.000    2643.103406  170511        168251
      Superior Completo            3000.000    3990.042331  242404        238842
 Pós-Graduação completa            4361.895    5994.859283   33303         32774
               Mestrado            4645.000    63

## 8. Enriquecimento com IA — agrupamento de CBOs em macrofamílias

São **2.207 CBOs distintos**. O guia exige agrupá-los em **macrofamílias** com IA
Generativa, para responder: *o gargalo está em funções operacionais de base ou em
cargos técnicos/engenharia?*

Estratégia robusta e reprodutível:
1. **Baseline determinístico** pela estrutura do próprio CBO-2002 (o 1º dígito do código
   define o Grande Grupo ocupacional). Isso funciona **sem rede** e já entrega macrofamílias.
2. **Refino opcional via LLM** (OpenAI) para nomear/ajustar famílias a partir dos CBOs mais
   frequentes. Ativado apenas se `OPENAI_API_KEY` estiver definida.


In [14]:
# 8.1 Baseline determinístico: Grande Grupo CBO (1º dígito)
GRANDE_GRUPO_CBO = {
    "0": "Forças Armadas / Segurança",
    "1": "Dirigentes e Gerentes",
    "2": "Profissionais de nível superior (Técnico/Engenharia)",
    "3": "Técnicos de nível médio",
    "4": "Trabalhadores de apoio administrativo",
    "5": "Serviços, vendas e comércio",
    "6": "Agropecuários e florestais",
    "7": "Produção de bens e serviços industriais (Operacional)",
    "8": "Produção de bens e serviços industriais (Operacional)",
    "9": "Manutenção e reparação",
}
cbo = dfm["cbo2002ocupação"].astype(str).str.zfill(6)
dfm["cbo_grande_grupo"] = cbo.str[0].map(GRANDE_GRUPO_CBO).fillna("Não classificado")

macro = (dfm.groupby("cbo_grande_grupo")
           .agg(saldo=("saldo","sum"),
                admissoes=("admissao","sum"),
                desligamentos=("desligamento","sum"),
                grau_medio=("grau_ord","mean"),
                pct_superior=("superior","mean"),
                n_cbos=("cbo2002ocupação","nunique"),
                **SAL_AGG)
           .reset_index().sort_values("saldo", ascending=False))
print(macro.to_string(index=False))

# Mesmo agrupamento, só nos setores industriais (B, C, D, F): o gargalo da INDÚSTRIA é
# operacional ou técnico?
macro_ind = (dfm[dfm["seção"].isin(SETORES_INDUSTRIAIS)].groupby("cbo_grande_grupo")
               .agg(saldo=("saldo","sum"),
                    admissoes=("admissao","sum"),
                    desligamentos=("desligamento","sum"),
                    grau_medio=("grau_ord","mean"),
                    pct_superior=("superior","mean"),
                    n_cbos=("cbo2002ocupação","nunique"),
                    **SAL_AGG)
               .reset_index().sort_values("saldo", ascending=False))
print("\nSó setores industriais:")
print(macro_ind.to_string(index=False))

                                     cbo_grande_grupo  saldo  admissoes  desligamentos  grau_medio  pct_superior  n_cbos  salario_medio  salario_mediano  salario_medio_s0  salario_mediano_s0  n_salario  n_salario_s0
                          Serviços, vendas e comércio  28457     503976         475519    6.509237      0.034201     164    2038.712500         2022.000       2072.258077             2024.00     979495        963639
                Trabalhadores de apoio administrativo  12769     317394         304625    6.975599      0.097566      94    2214.964314         2107.280       2217.895270             2107.28     622019        621197
Produção de bens e serviços industriais (Operacional)  10353     607391         597038    6.225609      0.015493     769    2491.785107         2250.000       2509.495334             2252.61    1204429       1195929
                           Agropecuários e florestais   1934      48052          46118    5.513932      0.006085     113    1921.376215 


Só setores industriais:
                                     cbo_grande_grupo  saldo  admissoes  desligamentos  grau_medio  pct_superior  n_cbos  salario_medio  salario_mediano  salario_medio_s0  salario_mediano_s0  n_salario  n_salario_s0
Produção de bens e serviços industriais (Operacional)   6232     418634         412402    6.154636      0.014973     728    2531.524040          2245.00       2536.465508             2245.00     831036        829417
                           Agropecuários e florestais   1308       3554           2246    5.812586      0.012414      55    2057.911186          2119.00       2058.621056             2119.50       5800          5798
                Trabalhadores de apoio administrativo    539      55082          54543    6.708296      0.082025      70    2273.692974          2172.60       2276.683555             2172.60     109625        109481
                          Serviços, vendas e comércio    307      31067          30760    6.378653      0.04200

In [15]:
# 8.2 (Opcional) Refino via LLM — nomeia macrofamílias a partir dos top CBOs
def refinar_macrofamilias_llm(top_n=60):
    api_key = os.environ.get("OPENAI_API_KEY")
    if not api_key:
        print("OPENAI_API_KEY não definida — pulando refino por LLM (baseline já aplicado).")
        return None
    try:
        from openai import OpenAI
    except ImportError:
        print("Pacote 'openai' não instalado (pip install openai). Usando baseline.")
        return None

    top = (dfm.groupby("cbo2002ocupação")
             .agg(saldo=("saldo","sum"), n=("saldo","size"))
             .reset_index().sort_values("n", ascending=False).head(top_n))
    lista = "\n".join(f'- CBO {r.cbo2002ocupação} (saldo {int(r.saldo)}, {int(r.n)} movs)'
                       for r in top.itertuples())
    prompt = (
        "Você é um especialista em mercado de trabalho industrial. Agrupe os códigos CBO-2002 "
        "abaixo em no máximo 8 MACROFAMÍLIAS ocupacionais úteis para a indústria catarinense "
        "(ex.: 'Operacional de produção', 'Técnico/Engenharia', 'Logística e transporte', "
        "'Administrativo', 'Manutenção'). Responda em JSON: lista de objetos "
        "{cbo, macrofamilia}.\n\n" + lista
    )
    client = OpenAI(api_key=api_key)
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    print("Resposta do LLM (recorte):")
    print(resp.choices[0].message.content[:1500])
    return resp.choices[0].message.content

_ = refinar_macrofamilias_llm()

OPENAI_API_KEY não definida — pulando refino por LLM (baseline já aplicado).


## 9. Exportação das tabelas-base (consumidas pelo dashboard)

Gravamos agregações enxutas em `outputs/` — são elas que o dashboard (Entregável 2) carrega.


In [16]:
saidas = {
    "agg_municipio.parquet":        terr,
    "agg_setor.parquet":            setor,
    "agg_municipio_setor.parquet":  terr_setor,
    "perfil_adm_desl.parquet":      perfil,
    "salario_por_grau.parquet":     sal_grau,
    "macrofamilias_cbo.parquet":    macro,
    "macrofamilias_industria.parquet": macro_ind,
    "agg_regiao.parquet":           regioes,
}
for nome, tabela in saidas.items():
    caminho = os.path.join(OUTPUTS, nome)
    tabela.to_parquet(caminho, index=False)
    print(f"gravado: {caminho}  ({tabela.shape[0]} linhas x {tabela.shape[1]} cols)")

print("\nPipeline concluído. Tabelas prontas para o dashboard.")

gravado: ../outputs/agg_municipio.parquet  (295 linhas x 41 cols)
gravado: ../outputs/agg_setor.parquet  (22 linhas x 17 cols)
gravado: ../outputs/agg_municipio_setor.parquet  (4425 linhas x 20 cols)
gravado: ../outputs/perfil_adm_desl.parquet  (24 linhas x 9 cols)
gravado: ../outputs/salario_por_grau.parquet  (12 linhas x 10 cols)
gravado: ../outputs/macrofamilias_cbo.parquet  (9 linhas x 13 cols)
gravado: ../outputs/macrofamilias_industria.parquet  (8 linhas x 13 cols)
gravado: ../outputs/agg_regiao.parquet  (31 linhas x 22 cols)

Pipeline concluído. Tabelas prontas para o dashboard.


## 10. Como ler os resultados (resumo analítico)

- **Expansão acelerada**: municípios/setores com `saldo_por_10k` alto e crescente.
- **Apagão de mão de obra**: `giro_por_10k` alto com `saldo` baixo/negativo — reposição
  constante sem crescimento líquido.
- **Causa-raiz (renda/escolaridade)**:
  - salário setorial abaixo da média + saldo negativo → **desatratividade salarial**;
  - vagas exigindo grau alto em região de baixa `escolarizacao`/IDHM → **escolaridade não atendida**;
  - salário estável apesar de maior exigência de instrução → **compressão salarial**.
- **IA/CBO**: a macrofamília com saldo negativo mostra se o apagão é **operacional** (grupos 7/8)
  ou **técnico/engenharia** (grupo 2) — base direta para as recomendações do pitch.

> **Limites da análise:** saldo e giro usam o próprio CAGED (fluxo, não estoque de vínculos);
> IDHM é de 2010; salários declarados podem conter outliers (filtrar no dashboard se necessário).
